# Microsoft Foundry — Provisionamento dos 5 agentes da workshop

Este notebook cria (ou atualiza) **5 agentes** no projeto `proj-ai-framework`
usando a API nova de **agent versions** do `azure-ai-projects` 1.1+.

| # | Agente                              | Knowledge Base                | Papel                                                              |
|---|-------------------------------------|-------------------------------|--------------------------------------------------------------------|
| 1 | `especialista-produtos`             | `telecom-products`            | Tira dúvidas sobre planos, fibra, dispositivos                     |
| 2 | `especialista-regulamentos`         | `internal-regulations`        | Responde dúvidas de RH, SegInfo e LGPD                             |
| 3 | `especialista-suporte-tecnico`      | `telecom-products`            | Diagnóstico de conectividade, modem, 5G                            |
| 4 | `especialista-vendas`               | `telecom-products`            | Recomendação de planos por perfil + objeções                       |
| 5 | `orquestrador-atendimento`          | (sem KB; só prompt)           | Identifica intenção e direciona ao especialista correto            |

Os 4 especialistas usam **Foundry IQ Knowledge** apontando para os índices criados
em `provision_indexes.ipynb` via `AzureAISearchTool`. O orquestrador é um prompt
agent que, dada a pergunta do usuário, retorna o nome do especialista a chamar
(padrão *router*; substitui o antigo `ConnectedAgentTool` que foi removido do SDK).

**Pré-requisitos**
- Connection `ai-search-ai-framework-conn` existe no projeto (Entra ID auth).
- A MSI do projeto Foundry tem `Search Index Data Reader` no AI Search.
- A identidade logada tem `Azure AI Project Manager` no projeto.

In [1]:
%pip install --quiet --upgrade "azure-ai-projects>=1.1.0" "azure-identity>=1.17.0" "python-dotenv>=1.0.1"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

ENV_PATH = Path.cwd() / ".env"
if not ENV_PATH.exists():
    raise FileNotFoundError(f".env not found at {ENV_PATH}.")
load_dotenv(ENV_PATH, override=True)

PROJECT_ENDPOINT  = os.environ["AZURE_AI_PROJECT_ENDPOINT"].rstrip("/")
SEARCH_ENDPOINT   = os.environ["AZURE_SEARCH_ENDPOINT"]
INDEX_TELECOM     = os.environ.get("AZURE_SEARCH_INDEX_TELECOM", "telecom-products")
INDEX_REGULATIONS = os.environ.get("AZURE_SEARCH_INDEX_REGULATIONS", "internal-regulations")
CHAT_MODEL        = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4.1-mini")

NAME_PRODUTOS     = os.environ.get("AZURE_AI_AGENT_PRODUTOS",     "especialista-produtos")
NAME_REGULAMENTOS = os.environ.get("AZURE_AI_AGENT_REGULAMENTOS", "especialista-regulamentos")
NAME_SUPORTE      = os.environ.get("AZURE_AI_AGENT_SUPORTE",      "especialista-suporte-tecnico")
NAME_VENDAS       = os.environ.get("AZURE_AI_AGENT_VENDAS",       "especialista-vendas")
NAME_ORCH         = os.environ.get("AZURE_AI_AGENT_ORCHESTRATOR", "orquestrador-atendimento")

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
print("Project endpoint:", PROJECT_ENDPOINT)
print("Chat model     :", CHAT_MODEL)

Project endpoint: https://foundry-ai-framework.services.ai.azure.com/api/projects/proj-ai-framework
Chat model     : gpt-4.1-mini


In [3]:
# A connection ai-search-ai-framework-conn deve existir no projeto Foundry.
CONNECTION_NAME = "ai-search-ai-framework-conn"

conn = next((c for c in project_client.connections.list() if c.name == CONNECTION_NAME), None)
if conn is None:
    raise RuntimeError(
        f"Connection '{CONNECTION_NAME}' não encontrada. Crie via portal "
        "(Project -> Connected resources -> + AI Search, Entra ID auth) ou via REST."
    )
SEARCH_CONN_ID = conn.id
print(f"SEARCH_CONN_ID = {SEARCH_CONN_ID}")

SEARCH_CONN_ID = /subscriptions/5d2aeade-0354-4dc5-b842-6aa61d1e0a40/resourceGroups/rg-framework-ai-contoso/providers/Microsoft.CognitiveServices/accounts/foundry-ai-framework/projects/proj-ai-framework/connections/ai-search-ai-framework-conn


## 2. Helpers — upsert idempotente e tool de busca

In [4]:
from azure.ai.projects.models import (
    PromptAgentDefinition,
    AzureAISearchTool,
    AzureAISearchToolResource,
    AISearchIndexResource,
    AzureAISearchQueryType,
)

def search_tool_for(index_name: str) -> AzureAISearchTool:
    return AzureAISearchTool(
        azure_ai_search=AzureAISearchToolResource(
            indexes=[AISearchIndexResource(
                project_connection_id=SEARCH_CONN_ID,
                index_name=index_name,
                query_type=AzureAISearchQueryType.SEMANTIC,
                top_k=5,
            )],
        ),
    )

def upsert_agent(*, name: str, instructions: str, description: str, tools=None):
    definition = PromptAgentDefinition(
        model=CHAT_MODEL,
        instructions=instructions,
        tools=tools or [],
        temperature=0.3,
    )
    # Try delete-then-create for idempotency (the new SDK uses immutable versions).
    try:
        project_client.agents.delete(agent_name=name)
        print(f"[reset] removed existing agent '{name}'")
    except Exception:
        pass
    created = project_client.agents.create_version(
        agent_name=name,
        definition=definition,
        description=description,
    )
    print(f"[create] {name} (version={getattr(created, 'version', '?')})")
    return created

## 3. Especialista — Produtos (índice `telecom-products`)

In [5]:
INSTR_PRODUTOS = """Você é o especialista de produtos da ConectaTel (telecom).

REGRAS:
- SEMPRE consulte o índice `telecom-products` antes de responder; nunca invente preços, SKUs ou prazos.
- Cite o SKU e o nome do produto entre parênteses ao mencionar um item.
- Se o cliente pedir algo fora do catálogo, diga claramente que não temos e sugira a alternativa mais próxima.
- Use português brasileiro, tom cordial e direto.

NUNCA:
- Prometa descontos, isenção de fidelidade ou condições não documentadas.
- Compare produtos com concorrentes nominalmente.
"""

ag_produtos = upsert_agent(
    name=NAME_PRODUTOS,
    description="Tira dúvidas sobre planos móveis, fibra, TV e dispositivos da ConectaTel.",
    instructions=INSTR_PRODUTOS,
    tools=[search_tool_for(INDEX_TELECOM)],
)

[create] especialista-produtos (version=1)


## 4. Especialista — Regulamentos (índice `internal-regulations`)

In [6]:
INSTR_REGULAMENTOS = """Você é o especialista de regulamentos internos da ConectaTel.

REGRAS:
- SEMPRE consulte o índice `internal-regulations` antes de responder.
- Cite o `policyId` e a versão da política como fonte (ex.: "conforme RH-003 v1.0").
- Quando a política não cobrir o cenário, diga isso e oriente o canal correto (RH, DPO, CSIRT).
- Use linguagem profissional e neutra. Nunca sugira burlar ou flexibilizar a política.

NUNCA:
- Invente cláusulas. Se não está na política, não existe.
- Forneça aconselhamento jurídico individual.
"""

ag_regulamentos = upsert_agent(
    name=NAME_REGULAMENTOS,
    description="Responde dúvidas sobre RH, Segurança da Informação e LGPD.",
    instructions=INSTR_REGULAMENTOS,
    tools=[search_tool_for(INDEX_REGULATIONS)],
)

[create] especialista-regulamentos (version=1)


## 5. Especialista — Suporte Técnico (índice `telecom-products`)

In [7]:
INSTR_SUPORTE = """Você é o especialista de suporte técnico da ConectaTel.

REGRAS:
- Antes de qualquer diagnóstico, identifique o produto/SKU consultando o índice `telecom-products`.
- Conduza um troubleshooting passo-a-passo (modem ligado? luz do PON? velocidade no Speedtest?).
- Para 5G/cobertura, informe que a verificação detalhada exige CEP + IMEI.
- Se o problema persistir após 3 passos, ofereça abrir um chamado.

NUNCA:
- Peça senha, CPF completo ou dados de cartão.
- Garanta SLA de retorno sem confirmar abertura de ticket.
"""

ag_suporte = upsert_agent(
    name=NAME_SUPORTE,
    description="Diagnóstico de conectividade, modem, Wi-Fi, 5G e dispositivos.",
    instructions=INSTR_SUPORTE,
    tools=[search_tool_for(INDEX_TELECOM)],
)

[create] especialista-suporte-tecnico (version=1)


## 6. Especialista — Vendas (índice `telecom-products`)

In [8]:
INSTR_VENDAS = """Você é o consultor de vendas da ConectaTel.

REGRAS:
- Use o índice `telecom-products` para recomendar planos.
- Faça 2 perguntas de qualificação (uso atual, orçamento) antes de recomendar 1 a 3 opções.
- Ao listar, mostre SKU, preço mensal, fidelidade e principais benefícios.
- Trate objeções com fatos do catálogo (ex.: 5G incluso, apps zero-rating).

NUNCA:
- Ofereça desconto fora da tabela ou condições não previstas.
- Pressione o cliente; respeite "não" como resposta final.
"""

ag_vendas = upsert_agent(
    name=NAME_VENDAS,
    description="Recomendação de planos por perfil e tratamento de objeções.",
    instructions=INSTR_VENDAS,
    tools=[search_tool_for(INDEX_TELECOM)],
)

[create] especialista-vendas (version=1)


## 7. Orquestrador — roteamento por intenção

A nova versão do SDK removeu `ConnectedAgentTool`. Aqui implementamos o
orquestrador como um *router prompt*: ele identifica a intenção e responde
com o nome do agente especialista a ser chamado em seguida pela aplicação.

In [9]:
INSTR_ORCH = f"""Você é o orquestrador de atendimento da ConectaTel.

Sua ÚNICA tarefa é classificar a mensagem do usuário e escolher EXATAMENTE UM
dos especialistas abaixo. Você NUNCA responde diretamente sobre produtos,
regulamentos ou suporte — você apenas roteia.

ESPECIALISTAS DISPONÍVEIS:
- "{NAME_PRODUTOS}"      → dúvidas sobre catálogo (planos, fibra, dispositivos, preço, fidelidade)
- "{NAME_VENDAS}"        → cliente quer contratar, comparar planos para si, pedir recomendação
- "{NAME_SUPORTE}"       → problema técnico, internet lenta, sem sinal, modem, configuração
- "{NAME_REGULAMENTOS}"  → políticas internas (RH, Segurança da Informação, LGPD, privacidade)

FORMATO DE RESPOSTA (obrigatório, JSON em uma linha):
{{"agent": "<nome-do-especialista>", "reason": "<motivo curto em PT-BR>"}}

Não inclua texto adicional fora do JSON.
"""

ag_orch = upsert_agent(
    name=NAME_ORCH,
    description="Classifica intenção e roteia para o especialista correto.",
    instructions=INSTR_ORCH,
    tools=[],
)

[create] orquestrador-atendimento (version=1)


## 8. Smoke test — listar agentes criados

In [10]:
print("Agents no projeto:")
for a in project_client.agents.list():
    print(f"  - {a.name}")

Agents no projeto:


  - orquestrador-atendimento
  - especialista-vendas
  - especialista-suporte-tecnico
  - especialista-regulamentos
  - especialista-produtos
